# Day 1 Sample Notebook: Exploring the OMOP CDM and Standardized Vocabularies

**OHDSI / OMOP Train-the-Trainer, ALS Therapy Development Institute**

This notebook is a runnable companion to the Day 1 module. It builds a tiny, synthetic OMOP Common Data Model (CDM) right here in the notebook using SQLite, so you can run every cell in Google Colab with no database credentials and no access to real patient data.

**All data in this notebook is synthetic and contains no protected health information.**

### Learning objectives
By the end of this notebook you will be able to:

1. Describe the core OMOP CDM clinical tables and how they connect.
2. Tell the difference between a standard and a non-standard concept.
3. Navigate the vocabulary using "Maps to", "Is a", and ancestor relationships.
4. Assemble a simple drug-exposure cohort with SQL.
5. Run a basic data-quality check for unmapped records.

### How to use it
Run the cells in order. Each section has a short explanation, a query you can run, and a **Your turn** prompt. Instructor notes with suggested answers are at the end.

> Trainer tip: this notebook mirrors the GUI steps in Athena and ATLAS. Have learners do the click-through in Athena first, then reproduce the same logic in SQL here.

## Step 0: Build a synthetic OMOP CDM

We create five familiar OMOP tables (`concept`, `concept_relationship`, `concept_ancestor`, `person`, `drug_exposure`, `condition_occurrence`) and load a small ALS-flavored sample: a SNOMED diagnosis concept for ALS, two RxNorm ALS drugs (riluzole and edaravone) with their ingredient concepts, and a handful of synthetic people.

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")
cur = con.cursor()

# --- Core vocabulary tables (simplified OMOP columns) ---
cur.executescript("""
CREATE TABLE concept (
    concept_id INTEGER PRIMARY KEY,
    concept_name TEXT,
    domain_id TEXT,
    vocabulary_id TEXT,
    concept_class_id TEXT,
    standard_concept TEXT,        -- 'S' standard, 'C' classification, NULL non-standard
    concept_code TEXT
);
CREATE TABLE concept_relationship (
    concept_id_1 INTEGER,
    concept_id_2 INTEGER,
    relationship_id TEXT
);
CREATE TABLE concept_ancestor (
    ancestor_concept_id INTEGER,
    descendant_concept_id INTEGER,
    min_levels_of_separation INTEGER
);

-- --- Clinical data tables ---
CREATE TABLE person (
    person_id INTEGER PRIMARY KEY,
    gender_concept_id INTEGER,
    year_of_birth INTEGER
);
CREATE TABLE condition_occurrence (
    condition_occurrence_id INTEGER PRIMARY KEY,
    person_id INTEGER,
    condition_concept_id INTEGER,
    condition_start_date TEXT
);
CREATE TABLE drug_exposure (
    drug_exposure_id INTEGER PRIMARY KEY,
    person_id INTEGER,
    drug_concept_id INTEGER,
    drug_exposure_start_date TEXT
);
""")
print("Schema created.")

In [ ]:
# --- Load synthetic vocabulary ---
concepts = [
    # concept_id, name, domain, vocab, class, standard, code
    (35748069, "Amyotrophic lateral sclerosis", "Condition", "SNOMED", "Clinical Finding", "S", "86044005"),
    (1314865,  "riluzole",            "Drug", "RxNorm", "Ingredient",    "S", "9468"),
    (1326727,  "edaravone",           "Drug", "RxNorm", "Ingredient",    "S", "1546020"),
    (19077572, "riluzole 50 MG Oral Tablet", "Drug", "RxNorm", "Clinical Drug", "S", "316158"),
    (40234834, "edaravone 30 MG injection",  "Drug", "RxNorm", "Clinical Drug", "S", "1546022"),
    (44819096, "ALS (legacy source code)",   "Condition", "ICD9CM", "4-dig billing", None, "335.20"),
    (8532,     "FEMALE", "Gender", "Gender", "Gender", "S", "F"),
    (8507,     "MALE",   "Gender", "Gender", "Gender", "S", "M"),
    (0,        "No matching concept", "Metadata", "None", "Undefined", None, "0"),
]
cur.executemany("INSERT INTO concept VALUES (?,?,?,?,?,?,?)", concepts)

# Maps to: legacy ICD9 ALS code maps to the standard SNOMED ALS concept
cur.executemany("INSERT INTO concept_relationship VALUES (?,?,?)", [
    (44819096, 35748069, "Maps to"),
    (19077572, 1314865,  "RxNorm has ing"),
    (40234834, 1326727,  "RxNorm has ing"),
])

# Ancestor: clinical drugs descend from their ingredients
cur.executemany("INSERT INTO concept_ancestor VALUES (?,?,?)", [
    (1314865, 19077572, 1),
    (1326727, 40234834, 1),
    (1314865, 1314865, 0),
    (1326727, 1326727, 0),
])

# People
cur.executemany("INSERT INTO person VALUES (?,?,?)", [
    (1, 8532, 1962), (2, 8507, 1955), (3, 8532, 1971), (4, 8507, 1948), (5, 8532, 1959),
])
cur.executemany("INSERT INTO condition_occurrence VALUES (?,?,?,?)", [
    (1, 1, 35748069, "2023-02-10"),
    (2, 2, 35748069, "2022-11-03"),
    (3, 3, 35748069, "2024-01-22"),
    (4, 4, 44819096, "2021-06-15"),   # recorded with legacy source code
])
cur.executemany("INSERT INTO drug_exposure VALUES (?,?,?,?)", [
    (1, 1, 19077572, "2023-03-01"),   # riluzole tablet
    (2, 2, 40234834, "2022-12-01"),   # edaravone
    (3, 3, 19077572, "2024-02-01"),   # riluzole tablet
    (4, 1, 40234834, "2023-09-01"),   # riluzole + edaravone (person 1)
    (5, 5, 0,        "2024-03-01"),   # unmapped drug (concept_id 0)
])
con.commit()
print("Synthetic data loaded.")

In [ ]:
def q(sql):
    """Run SQL against the synthetic CDM and return a tidy DataFrame."""
    return pd.read_sql_query(sql, con)

q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

## Step 1: Look at the clinical tables

In OMOP, each clinical fact lives in a domain table and points at a concept. A person can appear in many tables, joined on `person_id`. Notice that the clinical tables store concept *ids*, not text. The meaning comes from joining to `concept`.

In [ ]:
q("""
SELECT p.person_id, p.year_of_birth,
       c.concept_name AS condition,
       co.condition_start_date
FROM person p
JOIN condition_occurrence co ON co.person_id = p.person_id
JOIN concept c ON c.concept_id = co.condition_concept_id
ORDER BY p.person_id
""")

## Step 2: Standard versus non-standard concepts

The `standard_concept` flag is one of the most important ideas in OMOP. A value of `'S'` means the concept is the standard one you should analyze on. Source codes (for example legacy ICD-9) are non-standard (the flag is NULL) and must be mapped to a standard concept before analysis.

In [ ]:
q("""
SELECT concept_id, concept_name, vocabulary_id, concept_class_id,
       COALESCE(standard_concept, 'non-standard') AS standard_flag
FROM concept
WHERE domain_id IN ('Condition', 'Drug')
ORDER BY standard_flag DESC, vocabulary_id
""")

## Step 3: Follow the vocabulary relationships

Non-standard source codes connect to standard concepts through the **"Maps to"** relationship. Here we resolve the legacy ICD-9 ALS code to its standard SNOMED concept. This is exactly what an ETL does behind the scenes.

In [ ]:
q("""
SELECT src.concept_name  AS source_concept,
       src.vocabulary_id AS source_vocab,
       tgt.concept_name  AS standard_concept,
       tgt.vocabulary_id AS standard_vocab
FROM concept_relationship r
JOIN concept src ON src.concept_id = r.concept_id_1
JOIN concept tgt ON tgt.concept_id = r.concept_id_2
WHERE r.relationship_id = 'Maps to'
""")

## Step 4: Use the ancestor table to roll up drugs to ingredients

`concept_ancestor` lets you group all descendants of a concept without listing each one. To find everyone exposed to the **riluzole ingredient**, we ask for any drug that descends from concept 1314865, rather than naming every tablet and dose.

In [ ]:
q("""
SELECT ing.concept_name AS ingredient,
       d.concept_name    AS drug_product_taken
FROM concept_ancestor ca
JOIN concept ing ON ing.concept_id = ca.ancestor_concept_id
JOIN concept d   ON d.concept_id   = ca.descendant_concept_id
WHERE ing.concept_class_id = 'Ingredient'
ORDER BY ingredient
""")

## Step 5: Build a simple cohort

A cohort is the set of people who meet a definition. Here: **people exposed to the riluzole ingredient at any time.** We use the ancestor table so the definition is robust to whichever specific product was recorded.

In [ ]:
q("""
SELECT DISTINCT de.person_id
FROM drug_exposure de
JOIN concept_ancestor ca
     ON ca.descendant_concept_id = de.drug_concept_id
WHERE ca.ancestor_concept_id = 1314865   -- riluzole ingredient
ORDER BY de.person_id
""")

## Step 6: A basic data-quality check

A classic first data-quality check is the count of records that failed to map, which land on `concept_id = 0`. A high unmapped rate is a red flag for the ETL.

In [ ]:
q("""
SELECT
  SUM(CASE WHEN drug_concept_id = 0 THEN 1 ELSE 0 END) AS unmapped_rows,
  COUNT(*) AS total_rows,
  ROUND(100.0 * SUM(CASE WHEN drug_concept_id = 0 THEN 1 ELSE 0 END) / COUNT(*), 1)
    AS pct_unmapped
FROM drug_exposure
""")

## Your turn

Try these in new cells. Suggested answers are in the instructor notes below.

1. **Count cohort size.** Modify the Step 5 query to return the *number* of people exposed to riluzole, not the list.
2. **Edaravone cohort.** Write the equivalent cohort for the edaravone ingredient (concept 1326727).
3. **Two-drug users.** Find people who were exposed to *both* riluzole and edaravone ingredients.
4. **Age at first record.** For each person with an ALS condition, compute approximate age at `condition_start_date` (year of the date minus `year_of_birth`).

> Trainer note: encourage learners to first sketch the ATLAS cohort that matches question 3, then confirm the count here. Connecting the GUI logic to the SQL is the point of the lab.

## Instructor notes and answer key

<details>
<summary>Click to expand suggested answers</summary>

**1. Cohort size**
```sql
SELECT COUNT(DISTINCT de.person_id) AS n
FROM drug_exposure de
JOIN concept_ancestor ca ON ca.descendant_concept_id = de.drug_concept_id
WHERE ca.ancestor_concept_id = 1314865;
```

**2. Edaravone cohort** : same query with `ancestor_concept_id = 1326727`.

**3. Two-drug users**
```sql
SELECT de.person_id
FROM drug_exposure de
JOIN concept_ancestor ca ON ca.descendant_concept_id = de.drug_concept_id
WHERE ca.ancestor_concept_id IN (1314865, 1326727)
GROUP BY de.person_id
HAVING COUNT(DISTINCT ca.ancestor_concept_id) = 2;
```

**4. Approximate age**
```sql
SELECT p.person_id,
       CAST(strftime('%Y', co.condition_start_date) AS INTEGER) - p.year_of_birth AS approx_age
FROM person p
JOIN condition_occurrence co ON co.person_id = p.person_id
WHERE co.condition_concept_id = 35748069;
```

**Discussion prompts**
- Why use the ancestor table instead of listing every product concept?
- What happens to person 5 (the unmapped drug) in every cohort above, and why is that the correct behavior?
- In the real CDM, which other tables would you check to confirm an ALS diagnosis (for example `condition_occurrence` plus `observation` or `procedure_occurrence`)?

</details>

### Where to go next
- Athena vocabulary browser: https://athena.ohdsi.org/
- The Book of OHDSI, Chapters 4 and 5: https://ohdsi.github.io/TheBookOfOhdsi/
- Day 1 module and slides on this site.

*This sample uses synthetic data only. Replace the in-memory SQLite connection with your governed CDM connection (Databricks or DBeaver) to run the same queries against real data.*